### Direct Lab
Student's name: SETH Rattanak

In [1]:
from pyspark.sql import SparkSession
from operator import add

In [2]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("PageRank Example") \
    .getOrCreate()

# Create SparkContext
sc = spark.sparkContext

# Sample web graph data: (page, [list of pages it links to])
web_graph = [
    ("A", ["B", "C"]),
    ("B", ["C"]),
    ("C", ["A"]),
    ("D", ["C"])
]

# Parameters
damping_factor = 0.85 # Typical value for PageRank
max_iterations = 20
tolerance = 0.001

# Parallelize the web graph
links = sc.parallelize(web_graph)

# Initialize ranks (1.0 for each page)
pages = links.map(lambda x: x[0]).distinct()
N = pages.count() # Total number of pages
ranks = pages.map(lambda page: (page, 1.0))

# Calculate out-degree for each page
out_degree = links.map(lambda x: (x[0], len(x[1])))

def compute_contributions(urls_ranks):
    """Calculate contributions to linked pages"""
    url, (links_list, rank) = urls_ranks
    num_links = len(links_list)
    for link in links_list:
        yield (link, rank / num_links)

# Main PageRank iteration
for iteration in range(max_iterations):
    # Join links with ranks
    contribs = links.join(ranks)
    # Calculate contributions to each page's rank
    contribs = contribs.flatMap(compute_contributions)
    # Reduce by key and apply PageRank formula
    new_ranks = contribs.reduceByKey(add) \
    .mapValues(lambda rank: (1 - damping_factor) / N +
    damping_factor * rank)
    # Check for convergence
    rank_diffs = new_ranks.join(ranks) \
    .map(lambda x: abs(x[1][0] - x[1][1])) \
    .reduce(add)
    ranks = new_ranks
    print(f"Iteration {iteration + 1}: Total difference = {rank_diffs}")
    if rank_diffs < tolerance:
        print("Converged!")
        break
    
# Collect and display final results
final_ranks = ranks.collect()
print("\nFinal PageRank Scores:")
for (page, rank) in sorted(final_ranks):
    print(f"Page {page}: {rank:.4f}")
# Stop Spark session
spark.stop()

25/04/09 11:24:34 WARN Utils: Your hostname, Rattanaks-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.136.46 instead (on interface en0)
25/04/09 11:24:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/09 11:24:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Iteration 1: Total difference = 1.8125000000000002


Iteration 2: Total difference = 2.3906250000000004


Iteration 3: Total difference = 1.9507500000000007


Iteration 4: Total difference = 0.9442171875000003


...

Iteration 17: Total difference = 0.03619740454008108


Iteration 18: Total difference = 0.03076779385906897


Iteration 19: Total difference = 0.026152624780208605


Iteration 20: Total difference = 0.0222297310631773



Final PageRank Scores:
Page A: 0.3413
Page B: 0.1863
Page C: 0.3485


Explanation

1. Setup
- Creates a Spark session and context
- Defines a sample web graph as (page, [linked_pages])
- Sets PageRank parameters
2. Parallelization
- Uses sc.parallelize() to distribute the web graph across the cluster
- Creates RDDs (Resilient Distributed Datasets) for links and ranks
3. Map-Reduce Implementation
- map(): Transforms data (initial ranks, contributions)
- flatMap(): Computes contributions to linked pages
- reduceByKey(): Aggregates contributions for each page
- join(): Combines datasets
4. PageRank Algorithm
- Implements the power iteration method
- Uses the damping factor (0.85) and random jump probability
- Checks for convergence
5. Features
- Iterative computation until convergence or max iterations
- Distributed processing of the graph
- Convergence checking with tolerance